# Phase 1 (Step 2): The ReAct Pattern

Now that you understand Function Calling, let's understand how an Agent **thinks**.

Before frameworks like LangGraph existed, people built agents using the **ReAct** (Reason + Act) pattern. The idea is simple: You tell the LLM to write down its "Thought", followed by an "Action" (a tool call). The system then interrupts the LLM, executes the tool, and feeds the result back as an "Observation".

We are going to build a full Agent loop from scratch using just a Python `while` loop!

In [1]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
import re

llm = ChatOllama(model="qwen2.5:7b-instruct", temperature=0)

c:\Users\sujat\projects\AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Define the Tools
For this example, we'll give our agent a basic math calculator tool. Instead of using LangChain's `@tool` decorator, we'll just write a raw python function to show there's no magic.

In [2]:
def calculate(expression: str) -> str:
    """Evaluates a mathematical expression."""
    print(f"--> [SYSTEM] Executing calculator for: {expression}")
    try:
        # WARNING: eval() is dangerous in production! 
        # But for this local tutorial, it works perfectly.
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

# Our registry of available tools
known_tools = {
    "calculate": calculate
}

### 2. The ReAct System Prompt
This is the most important part. We give the LLM strict instructions on how it must format its output. We tell it to stop generating text as soon as it outputs an `Action`.

In [3]:
system_prompt = """
You are a logical AI assistant. You solve problems by thinking step-by-step and using tools if necessary.

You have access to the following tools:
- calculate: Evaluates a mathematical expression. Input should be a valid python math string (e.g. 10 * 5).

To use a tool, you MUST use the following format:
Thought: Describe what you need to do next.
Action: The name of the tool to use (e.g. calculate)
Action Input: The input to the tool (e.g. 10 * 5)
PAUSE

The system will then provide you with an:
Observation: <the result of the tool>

When you have the final answer, format it like this:
Thought: I have the final answer.
Final Answer: <your final answer here>
"""

### 3. The Agent Loop
Here is the core logic. We put the LLM in a loop. If it outputs "Action:", we run the tool and feed the "Observation" back into the loop. If it outputs "Final Answer:", we break the loop.

In [4]:
def run_agent(query: str, max_iterations=5):
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=query)
    ]
    
    for i in range(max_iterations):
        print(f"\n--- Iteration {i+1} ---")
        
        # 1. Ask the LLM to think/act
        response = llm.invoke(messages)
        text = response.content
        print(text)
        
        # Add LLM's response to history
        messages.append(AIMessage(content=text))
        
        # 2. Check if the LLM reached a Final Answer
        if "Final Answer:" in text:
            print("\n*** AGENT FINISHED ***")
            break
            
        # 3. Check if the LLM wants to use a Tool
        # We use simple regex to extract the Tool name and Input
        action_match = re.search(r"Action: (.*?)\nAction Input: (.*?)(?:\n|$)", text, re.IGNORECASE)
        
        if action_match:
            tool_name = action_match.group(1).strip()
            tool_input = action_match.group(2).strip()
            
            if tool_name in known_tools:
                # Execute the tool!
                result = known_tools[tool_name](tool_input)
                observation = f"Observation: {result}"
                print(f"\n{observation}")
                
                # Feed the observation back to the LLM for the next loop iteration
                messages.append(HumanMessage(content=observation))
            else:
                messages.append(HumanMessage(content=f"Observation: Tool {tool_name} not found."))
        else:
            # If the LLM didn't format correctly, tell it to try again
            messages.append(HumanMessage(content="Observation: Please format your response with Action/Action Input or Final Answer."))

### 4. Let's Test It!
We will ask a math question that requires multiple steps.

In [5]:
question = "What is 15 multiplied by 4, and then divided by 2?"
print(f"QUESTION: {question}")

run_agent(question)

QUESTION: What is 15 multiplied by 4, and then divided by 2?

--- Iteration 1 ---
Thought: I need to perform the calculation as described. The steps are multiplication first (15 * 4), followed by division (result / 2).
Action: calculate
Action Input: 15 * 4 / 2 PAUSE
--> [SYSTEM] Executing calculator for: 15 * 4 / 2 PAUSE

Observation: Error: invalid syntax (<string>, line 1)

--- Iteration 2 ---
Thought: It seems there was an error due to the way the expression was input. I need to ensure proper grouping with parentheses.
Action: calculate
Action Input: (15 * 4) / 2
--> [SYSTEM] Executing calculator for: (15 * 4) / 2

Observation: 30.0

--- Iteration 3 ---
Thought: The calculation has been successfully performed, and the result is 30.0. Since we are dealing with a multiplication and division operation, the answer should be an integer.
Thought: I have the final answer.
Final Answer: 30

*** AGENT FINISHED ***
